# **Working code for 3D and upto 2nd order-no partial observations**

In [29]:
import numpy as np
import pandas as pd
import argparse
import time
import sys
import os
import scipy.sparse.linalg as lg

from Cov import GaussianCovariance_generic
from Factors import (
    ExplicitKLFactorization,
    ImplicitKLFactorization
)

from meas import (
    PointMeasurement,
    dPointMeasurement,
    ddPointMeasurement, 
    dddPointMeasurement,
    ddddPointMeasurement

)

# from Cov_g import gaussian_cov_generic as g
from supernode_converter import (
    convert_measurements_to_list_of_dicts
)

In [30]:
def parse_commandline():
    parser = argparse.ArgumentParser()
    parser.add_argument("--sigma", help="lengthscale", type=float, default=2.15)
    parser.add_argument("--h", help="no_of_points", type=float, default=27)
    parser.add_argument("--nugget", type=float, default=1e-12)
    parser.add_argument("--rho", type=float, default=10.0)
    parser.add_argument("--k_neighbors", type=int, default=1)
    parser.add_argument("--threads",type=int,default=1) # threads for parallel processing
    parser.add_argument("--type",type=str,default="points") # used for specifying the way derivatives are ordered
    # use "None" for no derivatives, "points" for grouping derivatives by points, "meas" for grouping by derivatives
    parser.add_argument("--compare_exact", type=bool, default=False)

    #     return parser.parse_args()
    args, _ = parser.parse_known_args()
    return args


In [36]:
def build_symmetric(cov, M):
    N = len(M)
    output_matrix = np.zeros((N, N), dtype=np.float64)

    for i in range(N):
        for j in range(N):
            output_matrix[i, j] = cov(M[i], M[j])

    return output_matrix


def build_test(cov, te, tr):
    N = len(tr)
    M = len(te)
    output_matrix = np.zeros((M, N), dtype=np.float64)

    for i in range(M):
        for j in range(N):
            output_matrix[i, j] = cov(te[i], tr[j])

    return output_matrix

In [37]:
# building matrices
def get_Gram_matrices(cov, X_domain,X_test, sigma):
    d = X_domain.shape[0]
    N_domain = X_domain.shape[1]

    # different measurements for covariance calculation
    meas_0 = [PointMeasurement(X_domain[:, i]) for i in range(N_domain)]
    meas_d=[dPointMeasurement(X_domain[:, i],j) for j in range(d) for i in range(N_domain)]

    ind_2=[(0, 0), (0, 1), (1,1), (0,2), (1,2),(2,2)]
    meas_dd=[]
    for j in range(len(ind_2)):
        for i in range(N_domain):
            meas_dd.append(ddPointMeasurement(X_domain[:, i],ind_2[j]))

    measurements=meas_0+meas_d+meas_dd
    
    N_test= X_test.shape[1]
    meas_test=[PointMeasurement(X_test[:, i]) for i in range(N_test)]

    cov2= GaussianCovariance_generic(sigma)
#     measurements=convert_measurements_to_list_of_dicts(measurements)
    
    
    # this is build_symmetric
#     Theta_train=cov2.build_symmetric(measurements)
    Theta_train = build_symmetric(cov2, measurements)
    
#     meas_test=convert_measurements_to_list_of_dicts(meas_test)
    
    #this is build_test
#     Theta_test=cov2.build_test(meas_test,measurements)
    Theta_test = build_test(cov2, meas_test, measurements)
    


    # Code to build the matrix in pure python 
    # Theta_train = np.zeros((len(measurements),len(measurements)))
    # for i in range(len(measurements)):
    #     for j in range(i,len(measurements)):
    #         Theta_train[i,j]=cov(measurements[i],measurements[j])
    #         if i != j:
    #             Theta_train[j, i] = Theta_train[i, j]

    # Theta_test = np.zeros((len(meas_test),len(measurements)))
    # for i in range(len(meas_test)):
    #     for j in range(len(measurements)):
    #         Theta_test[i,j]=cov(meas_test[i],measurements[j])


    return Theta_train,Theta_test

In [38]:
def iterGPR_exact(cov, X_domain,X_test, sol_init, sigma, nugget: float):

    Theta_train, Theta_test = get_Gram_matrices(cov, X_domain,X_test, sigma)

    weights=np.linalg.solve((Theta_train + nugget * np.eye(Theta_train.shape[0])),sol_init)
    v=Theta_test @ weights
    return v

In [39]:
def fast_gpt_cg(cov,X_dom,X_test,rhs_with_der, sol_init,nugget, sigma, rho=4.0,k_neighbors=3,lamb=1.5,alpha=1.0,N_threads=1):
    N=np.shape(X_dom)[1]

    d=np.shape(X_dom)[0]

    meas_0=[PointMeasurement(X_dom[:, i]) for i in range(N)]
    meas_d=[dPointMeasurement(X_dom[:, i],j) for j in range(d) for i in range(N)]

    ind_2=[(0, 0), (0, 1), (1,1), (0,2), (1,2),(2,2)]
    meas_dd=[]
    for j in range(len(ind_2)):
        for i in range(N):
            meas_dd.append(ddPointMeasurement(X_dom[:, i],ind_2[j]))

    meas=[meas_0,meas_d,meas_dd]

    if args.type=="points":
        implicit_factor,_,_=ImplicitKLFactorization.implicit_kl_factorization_for_d(cov,meas,rho,k_neighbors)
    elif args.type=="meas":
        implicit_factor,_,_=ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(cov,meas,rho,k_neighbors)
    else:
        print("Incorrect ordering type")
    explicit_factor=ExplicitKLFactorization.Explicit_from_implicit(implicit_factor,nugget=nugget,N_threads=N_threads)

    U=explicit_factor.U
    L = U.transpose().tocsc() # Transpose and convert to Compressed Sparse Column format
    P = explicit_factor.P


    rhs_now=rhs_with_der
    
    Θinv_rhs=sol_init
    Θinv_rhs[P]=U@(L @ rhs_now[P])


    # measurements for testing
    N_test=np.shape(X_test)[1]
    meas_test=[PointMeasurement(X_test[:, i]) for i in range(N_test)]

    # build theta (prediction, train)
    meas_train=meas_0+meas_d+meas_dd

    reorde_meas_trian=[meas_train[i] for i in P]

#     reorde_meas_trian=convert_measurements_to_list_of_dicts(reorde_meas_trian)
#     meas_test=convert_measurements_to_list_of_dicts(meas_test)

#     cov2=g(cov.length_scale)
    cov2 = GaussianCovariance_generic(sigma)

#     Theta_test=cov2.build_test(meas_test,reorde_meas_trian)
    Theta_test = build_test(cov2, meas_test, reorde_meas_trian)

    # Code for pure python
    # Theta_test=np.zeros((len(meas_test),len(reorde_meas_trian)))
    # for i in range(len(meas_test)):
    #     for j in range(len(reorde_meas_trian)):
    #         Theta_test[i,j]=cov(meas_test[i],reorde_meas_trian[j])


    

    y_predicted=np.zeros(len(meas_test))

    y_predicted=Theta_test @ Θinv_rhs[P]

    return y_predicted

In [40]:
if __name__=="__main__":

    args = parse_commandline()
    der_indices=[[0], [1],[2], [3], [1, 1], [1, 2], [2, 2], [1, 3], [2, 3], [3, 3]] # indices for derivatives
    # Reading initial training set
    n=int(args.h)
    sigma = float(args.sigma)
    file=pd.read_csv(f"./train_G3_{n}.csv")
    data=np.zeros((3,n))
    data[0,:]=np.array(file['X0']).flatten()
    data[1,:]=np.array(file['X1']).flatten()
    data[2,:]=np.array(file['X2']).flatten()
    X_domain=data.T
    truth_with_der=np.zeros(n*len(der_indices))
    for i in range(len(der_indices)):
        truth_with_der[i*n:(i*n)+n]=np.array(file[str(der_indices[i])]).flatten()
    

    test_file=pd.read_csv("./test_G3.csv")
    data_test=np.zeros((3,test_file.shape[0]))
    data_test[0,:]=np.array(test_file['X0']).flatten()
    data_test[1,:]=np.array(test_file['X1']).flatten()
    data_test[2,:]=np.array(test_file['X2']).flatten()
    X_test=data_test.T
    test_truth=np.zeros(test_file.shape[0])
    test_truth=np.array(test_file['[0]']).flatten()
    

    N_domain=np.shape(truth_with_der)

    lengthscale = args.sigma
    cov = GaussianCovariance_generic(lengthscale)

    sol_init=np.zeros(N_domain)
    nugget = args.nugget

    rho=args.rho
    k_neighbors = args.k_neighbors
    N_threads=args.threads
    sol_1=fast_gpt_cg(cov,data,data_test, truth_with_der,sol_init,nugget, sigma, rho=rho,k_neighbors=k_neighbors,lamb=1.5,alpha=1.0,N_threads=N_threads)


    print("test_mse",np.mean((test_truth - sol_1)**2))

    if args.compare_exact==True:
        sol_exact=iterGPR_exact(cov,data,data_test,truth_with_der, sigma, nugget)
        print("test_exact",np.mean((test_truth - sol_exact)**2))

test_mse 2.1956803977428134e-05


# **Working code for 3D and upto 1st order-partial observations:**

# **All 0 order available but only partial 1st order available-with conditioning**

In [3]:
# import numpy as np
# import pandas as pd
# import time
# import argparse

# from Cov import GaussianCovariance_generic
# from Factors import (
#     ExplicitKLFactorization,
#     ImplicitKLFactorization
# )

# from meas import (
#     PointMeasurement,
#     dPointMeasurement
# )

# # =========================================================
# # Args
# # =========================================================

# def parse_args():
#     parser = argparse.ArgumentParser()

#     parser.add_argument("--sigma", type=float, default=2.15)
#     parser.add_argument("--h", type=int, default=27)
#     parser.add_argument("--nugget", type=float, default=1e-12)
#     parser.add_argument("--rho", type=float, default=10.0)
#     parser.add_argument("--k_neighbors", type=int, default=1)
#     parser.add_argument("--threads", type=int, default=1)
#     parser.add_argument("--type", type=str, default="points")
#     #     return parser.parse_args()
#     args, _ = parser.parse_known_args()
#     return args
# # =========================================================
# # Partial gradient mask
# # =========================================================

# def generate_grad_mask(d, N, missing_rate, seed=0):
#     rng = np.random.default_rng(seed)
#     return rng.random((d, N)) > missing_rate


# # =========================================================
# # Measurement builder
# # =========================================================

# def build_measurements_partial(X, grad_mask):
#     d, N = X.shape

#     meas_0 = [
#         PointMeasurement(X[:, i])
#         for i in range(N)
#     ]

#     meas_d = []
#     for j in range(d):
#         for i in range(N):
#             if grad_mask[j, i]:
#                 meas_d.append(
#                     dPointMeasurement(X[:, i], j)
#                 )

#     return meas_0 + meas_d


# # =========================================================
# # Target builder
# # =========================================================

# def build_targets_partial(Y_full, grad_mask, N, d):

#     y_list = []

#     # 0th order
#     y_list.append(Y_full[0:N])

#     # 1st order
#     offset = N
#     for j in range(d):
#         yj = Y_full[offset:offset+N]

#         for i in range(N):
#             if grad_mask[j, i]:
#                 y_list.append([yj[i]])

#         offset += N

#     return np.concatenate(y_list)


# # =========================================================
# # Kernel builders
# # =========================================================

# def build_symmetric(cov, M):
#     N = len(M)
#     K = np.zeros((N, N))

#     for i in range(N):
#         for j in range(N):
#             K[i, j] = cov(M[i], M[j])

#     return K


# def build_test(cov, te, tr):
#     M = len(te)
#     N = len(tr)

#     K = np.zeros((M, N))

#     for i in range(M):
#         for j in range(N):
#             K[i, j] = cov(te[i], tr[j])

#     return K


# # =========================================================
# # Sparse GP
# # =========================================================

# def sparse_gp(cov, X_train, X_test, y_train,
#               grad_mask, nugget, rho, k_neighbors, ordering, threads):

#     d, N = X_train.shape

#     meas_0 = [
#         PointMeasurement(X_train[:, i])
#         for i in range(N)
#     ]

#     meas_d = []
#     for j in range(d):
#         for i in range(N):
#             if grad_mask[j, i]:
#                 meas_d.append(
#                     dPointMeasurement(X_train[:, i], j)
#                 )

#     meas = [meas_0, meas_d]

#     if ordering == "points":
#         implicit, _, _ = ImplicitKLFactorization.\
#             implicit_kl_factorization_for_d(cov, meas, rho, k_neighbors)
#     else:
#         implicit, _, _ = ImplicitKLFactorization.\
#             implicit_kl_factorization_for_d_V2(cov, meas, rho, k_neighbors)

#     explicit = ExplicitKLFactorization.Explicit_from_implicit(
#         implicit,
#         nugget=nugget,
#         N_threads=threads
#     )

#     U = explicit.U
#     L = U.transpose().tocsc()
#     P = explicit.P

#     rhs = np.zeros_like(y_train)
#     rhs[P] = U @ (L @ y_train[P])

#     meas_train = meas_0 + meas_d
#     reordered = [meas_train[i] for i in P]

#     meas_test = [
#         PointMeasurement(X_test[:, i])
#         for i in range(X_test.shape[1])
#     ]

#     Ks = build_test(cov, meas_test, reordered)

#     return Ks @ rhs[P]


# # =========================================================
# # RMSE
# # =========================================================

# def rmse(y_true, y_pred):
#     return np.sqrt(np.mean((y_true - y_pred) ** 2))


# # =========================================================
# # MAIN
# # =========================================================

# if __name__ == "__main__":

#     args = parse_args()

#     train = pd.read_csv(f"./train_G3_{args.h}.csv")
#     test = pd.read_csv("./test_G3.csv")

#     X_full = np.vstack([
#         train["X0"].values,
#         train["X1"].values,
#         train["X2"].values
#     ])

#     X_test = np.vstack([
#         test["X0"].values,
#         test["X1"].values,
#         test["X2"].values
#     ])

#     y_test = test["[0]"].values

#     # Only 0th + 1st derivatives
#     Y_full = np.concatenate([
#         train["[0]"].values,
#         train["[1]"].values,
#         train["[2]"].values,
#         train["[3]"].values
#     ])

#     cov = GaussianCovariance_generic(args.sigma)

#     results = []

#     missing_rates = [0.0, 0.25, 0.5, 0.75]

#     for mr in missing_rates:

#         print(f"\nMissing rate = {mr}")

#         for n in range(5, args.h + 1):

#             X_now = X_full[:, :n]
#             d = X_now.shape[0]

#             Y_now = np.concatenate([
#                 Y_full[i*args.h:i*args.h+n]
#                 for i in range(1 + d)
#             ])

#             grad_mask = generate_grad_mask(d, n, mr)

#             y_train = build_targets_partial(Y_now, grad_mask, n, d)


#             # Sparse
#             t0 = time.time()
#             y_s = sparse_gp(
#                 cov, X_now, X_test,
#                 y_train, grad_mask,
#                 args.nugget, args.rho,
#                 args.k_neighbors,
#                 args.type, args.threads
#             )
#             t_sparse = time.time() - t0

#             results.append({
#                 "N": n,
#                 "missing_rate": mr,
#                 "rmse_sparse": rmse(y_test, y_s),
#                 "time_sparse": t_sparse
#             })

#     df = pd.DataFrame(results)
#     df.to_csv("partial_derivatives_experiment.csv", index=False)

#     print("\nSaved: partial_derivatives_experiment.csv")

In [2]:
# import numpy as np
# import pandas as pd
# import time
# import argparse

# from Cov import GaussianCovariance_generic
# from Factors import (
#     ExplicitKLFactorization,
#     ImplicitKLFactorization
# )

# from meas import (
#     PointMeasurement,
#     dPointMeasurement
# )

# # =========================================================
# # Args
# # =========================================================

# def parse_args():
#     parser = argparse.ArgumentParser()

#     parser.add_argument("--sigma", type=float, default=2.15)
#     parser.add_argument("--h", type=int, default=27)   # FIXED N
#     parser.add_argument("--nugget", type=float, default=1e-12)
#     parser.add_argument("--rho", type=float, default=10.0)
#     parser.add_argument("--k_neighbors", type=int, default=1)
#     parser.add_argument("--threads", type=int, default=1)
#     parser.add_argument("--type", type=str, default="points")

#     args, _ = parser.parse_known_args()
#     return args


# # =========================================================
# # Mask
# # =========================================================

# def generate_grad_mask(d, N, missing_rate, seed=0):
#     rng = np.random.default_rng(seed)
#     return rng.random((d, N)) > missing_rate


# # =========================================================
# # Measurement builder (FIXED: consistent partial model)
# # =========================================================

# def build_measurements(X, grad_mask):
#     d, N = X.shape

#     meas_0 = [PointMeasurement(X[:, i]) for i in range(N)]

#     meas_d = [
#         dPointMeasurement(X[:, i], j)
#         for j in range(d)
#         for i in range(N)
#         if grad_mask[j, i]
#     ]

#     return meas_0 + meas_d


# # =========================================================
# # Targets (FIXED: consistent indexing)
# # =========================================================

# def build_targets(Y0, Yd, grad_mask, N, d):

#     y = []

#     # function values
#     y.append(Y0)

#     # derivatives (masked)
#     idx = 0
#     for j in range(d):
#         for i in range(N):
#             if grad_mask[j, i]:
#                 y.append([Yd[j, i]])
#             idx += 1

#     return np.concatenate(y)


# # =========================================================
# # Kernel
# # =========================================================

# def build_test(cov, te, tr):
#     M = len(te)
#     N = len(tr)
#     K = np.zeros((M, N))

#     for i in range(M):
#         for j in range(N):
#             K[i, j] = cov(te[i], tr[j])

#     return K


# # =========================================================
# # Sparse GP
# # =========================================================

# def sparse_gp(cov, X_train, X_test, y_train,
#               grad_mask, nugget, rho, k_neighbors, ordering, threads):

#     d, N = X_train.shape

#     meas_0 = [PointMeasurement(X_train[:, i]) for i in range(N)]

#     meas_d = [
#         dPointMeasurement(X_train[:, i], j)
#         for j in range(d)
#         for i in range(N)
#         if grad_mask[j, i]
#     ]

#     meas = [meas_0, meas_d]

#     if ordering == "points":
#         implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d(
#             cov, meas, rho, k_neighbors
#         )
#     else:
#         implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(
#             cov, meas, rho, k_neighbors
#         )

#     explicit = ExplicitKLFactorization.Explicit_from_implicit(
#         implicit,
#         nugget=nugget,
#         N_threads=threads
#     )

#     U = explicit.U
#     L = U.transpose().tocsc()
#     P = explicit.P

#     rhs = np.zeros_like(y_train)
#     rhs[P] = U @ (L @ y_train[P])

#     meas_train = meas_0 + meas_d
#     reordered = [meas_train[i] for i in P]

#     meas_test = [PointMeasurement(X_test[:, i]) for i in range(X_test.shape[1])]

#     Ktest = build_test(cov, meas_test, reordered)

#     return Ktest @ rhs[P]


# # =========================================================
# # RMSE
# # =========================================================

# def rmse(y_true, y_pred):
#     return np.sqrt(np.mean((y_true - y_pred) ** 2))


# # =========================================================
# # MAIN (FIXED: SINGLE N ONLY)
# # =========================================================

# if __name__ == "__main__":

#     args = parse_args()

#     train = pd.read_csv(f"./train_G3_{args.h}.csv")
#     test = pd.read_csv("./test_G3.csv")

#     X = np.vstack([
#         train["X0"].values,
#         train["X1"].values,
#         train["X2"].values
#     ])

#     X_test = np.vstack([
#         test["X0"].values,
#         test["X1"].values,
#         test["X2"].values
#     ])

#     y_test = test["[0]"].values

#     N = args.h
#     d = X.shape[0]

#     # 0th order
#     Y0 = train["[0]"].values[:N]

#     # 1st derivatives ONLY
#     Yd = np.vstack([
#         train["[1]"].values[:N],
#         train["[2]"].values[:N],
#         train["[3]"].values[:N]
#     ])

#     cov = GaussianCovariance_generic(args.sigma)

#     results = []

#     missing_rates = [0.0, 0.25, 0.5, 0.75]

#     print(f"Fixed N = {N}")

#     for mr in missing_rates:

#         grad_mask = generate_grad_mask(d, N, mr)

#         y_train = build_targets(Y0, Yd, grad_mask, N, d)

#         t0 = time.time()

#         y_pred = sparse_gp(
#             cov, X[:, :N], X_test,
#             y_train, grad_mask,
#             args.nugget, args.rho,
#             args.k_neighbors,
#             args.type, args.threads
#         )

#         t = time.time() - t0

#         results.append({
#             "N": N,
#             "missing_rate": mr,
#             "rmse_sparse": rmse(y_test, y_pred),
#             "time_sparse": t
#         })

#     df = pd.DataFrame(results)
#     df.to_csv("partial_derivatives_fixedN.csv", index=False)

#     print("Saved: partial_derivatives_fixedN.csv")

Fixed N = 27
Saved: partial_derivatives_fixedN.csv


In [15]:
import numpy as np
import pandas as pd
import time
import argparse

from Cov import GaussianCovariance_generic
from Factors import (
    ExplicitKLFactorization,
    ImplicitKLFactorization
)

from meas import (
    PointMeasurement,
    dPointMeasurement
)

# =========================================================
# Args
# =========================================================

def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--sigma", type=float, default=2.15)
    parser.add_argument("--h", type=int, default=27)
    parser.add_argument("--nugget", type=float, default=1e-10)
    parser.add_argument("--rho", type=float, default=10.0)
    parser.add_argument("--k_neighbors", type=int, default=1)
    parser.add_argument("--threads", type=int, default=1)
    parser.add_argument("--type", type=str, default="points")

    args, _ = parser.parse_known_args()
    return args


# =========================================================
# Mask
# =========================================================

def generate_grad_mask(d, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((d, N)) > missing_rate


# =========================================================
# Measurements
# =========================================================

def build_measurements(X, grad_mask):
    d, N = X.shape

    meas_0 = [PointMeasurement(X[:, i]) for i in range(N)]

    meas_d = [
        dPointMeasurement(X[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]

    return meas_0 + meas_d


# =========================================================
# Kernel helpers
# =========================================================

def build_symmetric(cov, M):
    N = len(M)
    K = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            K[i, j] = cov(M[i], M[j])

    return K


# =========================================================
# Condition number
# =========================================================

def compute_condition_number(cov, measurements, nugget):
    K = build_symmetric(cov, measurements)
    K = K + nugget * np.eye(K.shape[0])
    return np.linalg.cond(K)


# =========================================================
# Targets
# =========================================================

def build_targets(Y0, Yd, grad_mask, N, d):

    y = [Y0]

    for j in range(d):
        for i in range(N):
            if grad_mask[j, i]:
                y.append([Yd[j, i]])

    return np.concatenate(y)


# =========================================================
# Sparse GP
# =========================================================

def sparse_gp(cov, X_train, X_test, y_train,
              grad_mask, nugget, rho, k_neighbors, ordering, threads):

    d, N = X_train.shape

    meas_0 = [PointMeasurement(X_train[:, i]) for i in range(N)]

    meas_d = [
        dPointMeasurement(X_train[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]

    meas = [meas_0, meas_d]

    if ordering == "points":
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d(
            cov, meas, rho, k_neighbors
        )
    else:
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(
            cov, meas, rho, k_neighbors
        )

    explicit = ExplicitKLFactorization.Explicit_from_implicit(
        implicit,
        nugget=nugget,
        N_threads=threads
    )

    U = explicit.U
    L = U.transpose().tocsc()
    P = explicit.P

    rhs = np.zeros_like(y_train)
    rhs[P] = U @ (L @ y_train[P])

    meas_train = meas_0 + meas_d
    reordered = [meas_train[i] for i in P]

    meas_test = [PointMeasurement(X_test[:, i]) for i in range(X_test.shape[1])]

    Ktest = np.zeros((len(meas_test), len(reordered)))
    for i in range(len(meas_test)):
        for j in range(len(reordered)):
            Ktest[i, j] = cov(meas_test[i], reordered[j])

    return Ktest @ rhs[P]


# =========================================================
# RMSE
# =========================================================

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


# =========================================================
# MAIN (FIXED N EXPERIMENT + CONDITION NUMBER)
# =========================================================

if __name__ == "__main__":

    args = parse_args()

    train = pd.read_csv(f"./train_G3_{args.h}.csv")
    test = pd.read_csv("./test_G3.csv")

    X = np.vstack([
        train["X0"].values,
        train["X1"].values,
        train["X2"].values
    ])

    X_test = np.vstack([
        test["X0"].values,
        test["X1"].values,
        test["X2"].values
    ])

    y_test = test["[0]"].values

    N = args.h
    d = X.shape[0]

    Y0 = train["[0]"].values[:N]

    Yd = np.vstack([
        train["[1]"].values[:N],
        train["[2]"].values[:N],
        train["[3]"].values[:N]
    ])

    cov = GaussianCovariance_generic(args.sigma)

    results = []

    missing_rates = [0.0, 0.25, 0.5, 0.75]

    print(f"Fixed N = {N}")

    for mr in missing_rates:

        grad_mask = generate_grad_mask(d, N, mr)

        # ---- build measurements for condition number
        meas = build_measurements(X[:, :N], grad_mask)
        cond = compute_condition_number(cov, meas, args.nugget)

        y_train = build_targets(Y0, Yd, grad_mask, N, d)

        t0 = time.time()

        y_pred = sparse_gp(
            cov, X[:, :N], X_test,
            y_train, grad_mask,
            args.nugget, args.rho,
            args.k_neighbors,
            args.type, args.threads
        )

        t = time.time() - t0

        results.append({
            "N": N,
            "missing_rate": mr,
            "mse_sparse": mse(y_test, y_pred),
            "time_sparse": t,
            "cond_number": cond
        })

    df = pd.DataFrame(results)
    df.to_csv(f"partial_derivatives_fixed_with_cond_{N}_.csv", index=False)

    print("Saved: partial_derivatives_fixed_with_cond.csv")

Fixed N = 27
Saved: partial_derivatives_fixed_with_cond.csv


In [11]:
# import numpy as np
# import pandas as pd
# import time
# import argparse

# from Cov import GaussianCovariance_generic
# from Factors import (
#     ExplicitKLFactorization,
#     ImplicitKLFactorization
# )

# from meas import (
#     PointMeasurement,
#     dPointMeasurement
# )

# # =========================================================
# # Args
# # =========================================================

# def parse_args():
#     parser = argparse.ArgumentParser()

#     parser.add_argument("--sigma", type=float, default=2.15)
#     parser.add_argument("--h", type=int, default=64)
#     parser.add_argument("--nugget", type=float, default=1e-10)
#     parser.add_argument("--rho", type=float, default=10.0)
#     parser.add_argument("--k_neighbors", type=int, default=1)
#     parser.add_argument("--threads", type=int, default=1)
#     parser.add_argument("--type", type=str, default="points")

#     args, _ = parser.parse_known_args()
#     return args


# # =========================================================
# # Mask
# # =========================================================

# def generate_grad_mask(d, N, missing_rate, seed=0):
#     rng = np.random.default_rng(seed)
#     return rng.random((d, N)) > missing_rate


# # =========================================================
# # FIXED: consistent measurement construction
# # =========================================================

# def build_measurements_and_targets(X, train, grad_mask, N):
#     """
#     Builds BOTH:
#     - measurement list (kernel indexing)
#     - target vector (aligned exactly)
#     """

#     d = X.shape[0]

#     meas = []
#     y = []

#     # -------------------------
#     # 0th order (function values)
#     # -------------------------
#     for i in range(N):
#         meas.append(PointMeasurement(X[:, i]))
#         y.append(train["[0]"].values[i])

#     # -------------------------
#     # 1st order derivatives
#     # -------------------------
#     deriv_keys = ["[1]", "[2]", "[3]"]

#     for j, key in enumerate(deriv_keys):
#         vals = train[key].values

#         for i in range(N):
#             if grad_mask[j, i]:
#                 meas.append(dPointMeasurement(X[:, i], j))
#                 y.append(vals[i])

#     return meas, np.array(y)


# # =========================================================
# # Kernel
# # =========================================================

# def build_test(cov, te, tr):
#     M = len(te)
#     N = len(tr)

#     K = np.zeros((M, N))
#     for i in range(M):
#         for j in range(N):
#             K[i, j] = cov(te[i], tr[j])

#     return K


# # =========================================================
# # Sparse GP
# # =========================================================

# def sparse_gp(cov, X_train, X_test, meas_train, y_train,
#               grad_mask, nugget, rho, k_neighbors, ordering, threads):

#     d, N = X_train.shape

#     # split meas into groups for KL factorization
#     meas_0 = [m for m in meas_train if isinstance(m, PointMeasurement)]
#     meas_d = [m for m in meas_train if isinstance(m, dPointMeasurement)]

#     meas = [meas_0, meas_d]

#     if ordering == "points":
#         implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d(
#             cov, meas, rho, k_neighbors
#         )
#     else:
#         implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(
#             cov, meas, rho, k_neighbors
#         )

#     explicit = ExplicitKLFactorization.Explicit_from_implicit(
#         implicit,
#         nugget=nugget,
#         N_threads=threads
#     )

#     U = explicit.U
#     L = U.transpose().tocsc()
#     P = explicit.P

#     rhs = np.zeros_like(y_train)
#     rhs[P] = U @ (L @ y_train[P])

#     # reorder measurements consistently
#     reordered = [meas_train[i] for i in P]

#     meas_test = [PointMeasurement(X_test[:, i]) for i in range(X_test.shape[1])]

#     Ktest = build_test(cov, meas_test, reordered)

#     return Ktest @ rhs[P]


# # =========================================================
# # RMSE
# # =========================================================

# def rmse(y_true, y_pred):
#     return np.sqrt(np.mean((y_true - y_pred) ** 2))


# # =========================================================
# # MAIN EXPERIMENT (FIXED N)
# # =========================================================

# if __name__ == "__main__":

#     args = parse_args()

#     train = pd.read_csv(f"./train_G3_{args.h}.csv")
#     test = pd.read_csv("./test_G3.csv")

#     X = np.vstack([
#         train["X0"].values,
#         train["X1"].values,
#         train["X2"].values
#     ])

#     X_test = np.vstack([
#         test["X0"].values,
#         test["X1"].values,
#         test["X2"].values
#     ])

#     y_test = test["[0]"].values

#     N = args.h

#     cov = GaussianCovariance_generic(args.sigma)

#     results = []

#     missing_rates = [0.0, 0.25, 0.5, 0.75]

#     print(f"\nFixed N = {N}")

#     for mr in missing_rates:

#         # mask
#         grad_mask = generate_grad_mask(X.shape[0], N, mr)

#         #  FIXED: consistent measurement + target construction
#         meas_train, y_train = build_measurements_and_targets(
#             X[:, :N],
#             train,
#             grad_mask,
#             N
#         )

#         t0 = time.time()

#         y_pred = sparse_gp(
#             cov,
#             X[:, :N],
#             X_test,
#             meas_train,
#             y_train,
#             grad_mask,
#             args.nugget,
#             args.rho,
#             args.k_neighbors,
#             args.type,
#             args.threads
#         )

#         t = time.time() - t0

#         results.append({
#             "N": N,
#             "missing_rate": mr,
#             "rmse_sparse": rmse(y_test, y_pred),
#             "time_sparse": t
#         })

#     df = pd.DataFrame(results)
#     df.to_csv("partial_derivatives_fixed_64_clean.csv", index=False)

#     print("Saved: partial_derivatives_fixed_64_clean.csv")


Fixed N = 64
Saved: partial_derivatives_fixed_64_clean.csv


# **Working code for 3D and upto 2nd order-partial observations:**

# **All 0 order available but only fixed partial (0.5) 1st order missing, variable missing 2nd order-with conditioning**

In [ ]:
# import numpy as np
# import pandas as pd
# import argparse
# import time
# import sys
# import os
# import scipy.sparse.linalg as lg

# from Cov import GaussianCovariance_generic
# from Factors import (
#     ExplicitKLFactorization,
#     ImplicitKLFactorization
# )

# from meas import (
#     PointMeasurement,
#     dPointMeasurement,
#     ddPointMeasurement, 
#     dddPointMeasurement,
#     ddddPointMeasurement

# )

# # from Cov_g import gaussian_cov_generic as g
# from supernode_converter import (
#     convert_measurements_to_list_of_dicts
# )

# def parse_commandline():
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--sigma", help="lengthscale", type=float, default=2.15)
#     parser.add_argument("--h", help="no_of_points", type=float, default=27)
#     parser.add_argument("--nugget", type=float, default=1e-12)
#     parser.add_argument("--rho", type=float, default=10.0)
#     parser.add_argument("--k_neighbors", type=int, default=1)
#     parser.add_argument("--threads",type=int,default=1) # threads for parallel processing
#     parser.add_argument("--type",type=str,default="points") # used for specifying the way derivatives are ordered
#     # use "None" for no derivatives, "points" for grouping derivatives by points, "meas" for grouping by derivatives
#     parser.add_argument("--compare_exact", type=bool, default=False)


#     args, _ = parser.parse_known_args()
#     return args

# def build_symmetric(cov, M):
#     N = len(M)
#     output_matrix = np.zeros((N, N), dtype=np.float64)

#     for i in range(N):
#         for j in range(N):
#             output_matrix[i, j] = cov(M[i], M[j])

#     return output_matrix


# def build_test(cov, te, tr):
#     N = len(tr)
#     M = len(te)
#     output_matrix = np.zeros((M, N), dtype=np.float64)

#     for i in range(M):
#         for j in range(N):
#             output_matrix[i, j] = cov(te[i], tr[j])

#     return output_matrix

# # building matrices
# def get_Gram_matrices(cov, X_domain,X_test, sigma):
#     d = X_domain.shape[0]
#     N_domain = X_domain.shape[1]

#     # different measurements for covariance calculation
#     meas_0 = [PointMeasurement(X_domain[:, i]) for i in range(N_domain)]
#     meas_d=[dPointMeasurement(X_domain[:, i],j) for j in range(d) for i in range(N_domain)]

#     ind_2=[(0, 0), (0, 1), (1,1), (0,2), (1,2),(2,2)]
#     meas_dd=[]
#     for j in range(len(ind_2)):
#         for i in range(N_domain):
#             meas_dd.append(ddPointMeasurement(X_domain[:, i],ind_2[j]))

#     measurements=meas_0+meas_d+meas_dd
    
#     N_test= X_test.shape[1]
#     meas_test=[PointMeasurement(X_test[:, i]) for i in range(N_test)]

#     cov2= GaussianCovariance_generic(sigma)
# #     measurements=convert_measurements_to_list_of_dicts(measurements)
    
    
#     # this is build_symmetric
# #     Theta_train=cov2.build_symmetric(measurements)
#     Theta_train = build_symmetric(cov2, measurements)
    
# #     meas_test=convert_measurements_to_list_of_dicts(meas_test)
    
#     #this is build_test
# #     Theta_test=cov2.build_test(meas_test,measurements)
#     Theta_test = build_test(cov2, meas_test, measurements)
    


#     # Code to build the matrix in pure python 
#     # Theta_train = np.zeros((len(measurements),len(measurements)))
#     # for i in range(len(measurements)):
#     #     for j in range(i,len(measurements)):
#     #         Theta_train[i,j]=cov(measurements[i],measurements[j])
#     #         if i != j:
#     #             Theta_train[j, i] = Theta_train[i, j]

#     # Theta_test = np.zeros((len(meas_test),len(measurements)))
#     # for i in range(len(meas_test)):
#     #     for j in range(len(measurements)):
#     #         Theta_test[i,j]=cov(meas_test[i],measurements[j])


#     return Theta_train,Theta_test

# def iterGPR_exact(cov, X_domain,X_test, sol_init, sigma, nugget: float):

#     Theta_train, Theta_test = get_Gram_matrices(cov, X_domain,X_test, sigma)

#     weights=np.linalg.solve((Theta_train + nugget * np.eye(Theta_train.shape[0])),sol_init)
#     v=Theta_test @ weights
#     return v

# def fast_gpt_cg(cov,X_dom,X_test,rhs_with_der, sol_init,nugget, sigma, rho=4.0,k_neighbors=3,lamb=1.5,alpha=1.0,N_threads=1):
#     N=np.shape(X_dom)[1]

#     d=np.shape(X_dom)[0]

#     meas_0=[PointMeasurement(X_dom[:, i]) for i in range(N)]
#     meas_d=[dPointMeasurement(X_dom[:, i],j) for j in range(d) for i in range(N)]

#     ind_2=[(0, 0), (0, 1), (1,1), (0,2), (1,2),(2,2)]
#     meas_dd=[]
#     for j in range(len(ind_2)):
#         for i in range(N):
#             meas_dd.append(ddPointMeasurement(X_dom[:, i],ind_2[j]))

#     meas=[meas_0,meas_d,meas_dd]

#     if args.type=="points":
#         implicit_factor,_,_=ImplicitKLFactorization.implicit_kl_factorization_for_d(cov,meas,rho,k_neighbors)
#     elif args.type=="meas":
#         implicit_factor,_,_=ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(cov,meas,rho,k_neighbors)
#     else:
#         print("Incorrect ordering type")
#     explicit_factor=ExplicitKLFactorization.Explicit_from_implicit(implicit_factor,nugget=nugget,N_threads=N_threads)

#     U=explicit_factor.U
#     L = U.transpose().tocsc() # Transpose and convert to Compressed Sparse Column format
#     P = explicit_factor.P


#     rhs_now=rhs_with_der
    
#     Θinv_rhs=sol_init
#     Θinv_rhs[P]=U@(L @ rhs_now[P])


#     # measurements for testing
#     N_test=np.shape(X_test)[1]
#     meas_test=[PointMeasurement(X_test[:, i]) for i in range(N_test)]

#     # build theta (prediction, train)
#     meas_train=meas_0+meas_d+meas_dd

#     reorde_meas_trian=[meas_train[i] for i in P]

# #     reorde_meas_trian=convert_measurements_to_list_of_dicts(reorde_meas_trian)
# #     meas_test=convert_measurements_to_list_of_dicts(meas_test)

# #     cov2=g(cov.length_scale)
#     cov2 = GaussianCovariance_generic(sigma)

# #     Theta_test=cov2.build_test(meas_test,reorde_meas_trian)
#     Theta_test = build_test(cov2, meas_test, reorde_meas_trian)

#     # Code for pure python
#     # Theta_test=np.zeros((len(meas_test),len(reorde_meas_trian)))
#     # for i in range(len(meas_test)):
#     #     for j in range(len(reorde_meas_trian)):
#     #         Theta_test[i,j]=cov(meas_test[i],reorde_meas_trian[j])


    

#     y_predicted=np.zeros(len(meas_test))

#     y_predicted=Theta_test @ Θinv_rhs[P]

#     return y_predicted

# if __name__=="__main__":

#     args = parse_commandline()
#     der_indices=[[0], [1],[2], [3], [1, 1], [1, 2], [2, 2], [1, 3], [2, 3], [3, 3]] # indices for derivatives
#     # Reading initial training set
#     n=int(args.h)
#     sigma = float(args.sigma)
#     file=pd.read_csv(f"./train_G3_{n}.csv")
#     data=np.zeros((3,n))
#     data[0,:]=np.array(file['X0']).flatten()
#     data[1,:]=np.array(file['X1']).flatten()
#     data[2,:]=np.array(file['X2']).flatten()
#     X_domain=data.T
#     truth_with_der=np.zeros(n*len(der_indices))
#     for i in range(len(der_indices)):
#         truth_with_der[i*n:(i*n)+n]=np.array(file[str(der_indices[i])]).flatten()
    

#     test_file=pd.read_csv("./test_G3.csv")
#     data_test=np.zeros((3,test_file.shape[0]))
#     data_test[0,:]=np.array(test_file['X0']).flatten()
#     data_test[1,:]=np.array(test_file['X1']).flatten()
#     data_test[2,:]=np.array(test_file['X2']).flatten()
#     X_test=data_test.T
#     test_truth=np.zeros(test_file.shape[0])
#     test_truth=np.array(test_file['[0]']).flatten()
    

#     N_domain=np.shape(truth_with_der)

#     lengthscale = args.sigma
#     cov = GaussianCovariance_generic(lengthscale)

#     sol_init=np.zeros(N_domain)
#     nugget = args.nugget

#     rho=args.rho
#     k_neighbors = args.k_neighbors
#     N_threads=args.threads
#     sol_1=fast_gpt_cg(cov,data,data_test, truth_with_der,sol_init,nugget, sigma, rho=rho,k_neighbors=k_neighbors,lamb=1.5,alpha=1.0,N_threads=N_threads)


#     print("test_mse",np.mean((test_truth - sol_1)**2))

#     if args.compare_exact==True:
#         sol_exact=iterGPR_exact(cov,data,data_test,truth_with_der, sigma, nugget)
#         print("test_exact",np.mean((test_truth - sol_exact)**2))



In [41]:
import numpy as np
import pandas as pd
import time
import argparse

from Cov import GaussianCovariance_generic
from Factors import (
    ExplicitKLFactorization,
    ImplicitKLFactorization
)

from meas import (
    PointMeasurement,
    dPointMeasurement, ddPointMeasurement
)

# =========================================================
# Args
# =========================================================

def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--sigma", type=float, default=1.45)
    parser.add_argument("--h", type=int, default=27)
    parser.add_argument("--nugget", type=float, default=1e-8)
    parser.add_argument("--rho", type=float, default=10.0)
    parser.add_argument("--k_neighbors", type=int, default=1)
    parser.add_argument("--threads", type=int, default=1)
    parser.add_argument("--type", type=str, default="points")

    args, _ = parser.parse_known_args()
    return args


# =========================================================
# Mask
# =========================================================

# this mask is for gradient
def generate_grad_mask(d, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((d, N)) > missing_rate


# this mask is for hessians
def generate_hessian_mask(n_hess, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n_hess, N)) > missing_rate


# =========================================================
# Measurements
# =========================================================

def build_measurements(X, grad_mask, hess_mask):
    d, N = X.shape

    meas_0 = [PointMeasurement(X[:, i]) for i in range(N)]

    meas_d = [
        dPointMeasurement(X[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]
    

    ind_2=[(0, 0), (0, 1), (1,1), (0,2), (1,2),(2,2)]
    meas_dd=[]
    
#     for j in range(len(ind_2)):
#         for i in range(N):
#             meas_dd.append(ddPointMeasurement(X[:, i],ind_2[j]))

    
    for j in range(len(ind_2)):
        for i in range(N):
            if hess_mask[j, i]:
                meas_dd.append(ddPointMeasurement(X[:, i], ind_2[j]))

    return meas_0 + meas_d + meas_dd


# =========================================================
# Kernel helpers
# =========================================================

def build_symmetric(cov, M):
    N = len(M)
    K = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            K[i, j] = cov(M[i], M[j])

    return K


# =========================================================
# Condition number
# =========================================================

def compute_condition_number(cov, measurements, nugget):
    K = build_symmetric(cov, measurements)
    K = K + nugget * np.eye(K.shape[0])
    return np.linalg.cond(K)


# =========================================================
# Targets
# =========================================================

def build_targets(Y0, Yd, Ydd, grad_mask, hess_mask, N, d):

#     y = [Y0]

#     for j in range(d):
#         for i in range(N):
#             if grad_mask[j, i]:
#                 y.append([Yd[j, i]])

#     return np.concatenate(y)
    y = [Y0]
    for j in range(d):
        for i in range(N):
            if grad_mask[j, i]:
                y.append([Yd[j, i]])

    
#     for j in range(Ydd.shape[0]):
#         for i in range(N):
#             y.append([Ydd[j, i]])
    
    for j in range(Ydd.shape[0]):
        for i in range(N):
            if hess_mask[j, i]:
                y.append([Ydd[j, i]])

    return np.concatenate(y)


# =========================================================
# Sparse GP
# =========================================================

def sparse_gp(cov, X_train, X_test, y_train,
              grad_mask, hess_mask, nugget, rho, k_neighbors, ordering, threads):

    d, N = X_train.shape

    meas_0 = [PointMeasurement(X_train[:, i]) for i in range(N)]

    meas_d = [
        dPointMeasurement(X_train[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]
           
    
    ind_2=[(0, 0), (0, 1), (1,1), (0,2), (1,2),(2,2)]
    meas_dd=[]
    
    
#     for j in range(len(ind_2)):
#         for i in range(N):
#             meas_dd.append(ddPointMeasurement(X_train[:, i],ind_2[j]))     
    
    for j in range(len(ind_2)):
        for i in range(N):
            if hess_mask[j, i]:
                meas_dd.append(ddPointMeasurement(X_train[:, i], ind_2[j]))
    
    
    meas = [meas_0, meas_d, meas_dd]

    if ordering == "points":
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d(
            cov, meas, rho, k_neighbors
        )
    else:
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(
            cov, meas, rho, k_neighbors
        )

    explicit = ExplicitKLFactorization.Explicit_from_implicit(
        implicit,
        nugget=nugget,
        N_threads=threads
    )

    U = explicit.U
    L = U.transpose().tocsc()
    P = explicit.P
    print("this is P ", len(P))

    rhs = np.zeros_like(y_train)
    rhs[P] = U @ (L @ y_train[P])

    meas_train = meas_0 + meas_d + meas_dd
    print("this is meas_train ", len(meas_train))
    reordered = [meas_train[i] for i in P]

    meas_test = [PointMeasurement(X_test[:, i]) for i in range(X_test.shape[1])]

    Ktest = np.zeros((len(meas_test), len(reordered)))
    for i in range(len(meas_test)):
        for j in range(len(reordered)):
            Ktest[i, j] = cov(meas_test[i], reordered[j])

    return Ktest @ rhs[P]


# =========================================================
# RMSE
# =========================================================

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


# =========================================================
# MAIN (FIXED N EXPERIMENT + CONDITION NUMBER)
# =========================================================

if __name__ == "__main__":

    args = parse_args()

    train = pd.read_csv(f"./train_G3_{args.h}.csv")
    test = pd.read_csv("./test_G3.csv")

    X = np.vstack([
        train["X0"].values,
        train["X1"].values,
        train["X2"].values
    ])

    X_test = np.vstack([
        test["X0"].values,
        test["X1"].values,
        test["X2"].values
    ])

    y_test = test["[0]"].values

    N = args.h
    d = X.shape[0]

    Y0 = train["[0]"].values[:N]

    Yd = np.vstack([
        train["[1]"].values[:N],
        train["[2]"].values[:N],
        train["[3]"].values[:N]
    ])
    
    second_order = [
        "[1, 1]", "[1, 2]", "[2, 2]", "[1, 3]", "[2, 3]", "[3, 3]"
    ]

    Ydd = np.vstack([
        train[c].values[:N]
        for c in second_order
    ])
    
    cov = GaussianCovariance_generic(args.sigma)

    results = []

#     missing_rates = [0.0, 0.25, 0.5, 0.75]

    hessian_missing_rates = [0.0, 0.25, 0.5, 0.75]
    print(f"Fixed N = {N}")

#     for mr in missing_rates:

#         grad_mask = generate_grad_mask(d, N, 0.5)

#         # ---- build measurements for condition number
#         meas = build_measurements(X[:, :N], grad_mask)
#         cond = compute_condition_number(cov, meas, args.nugget)

#         y_train = build_targets(Y0, Yd, Ydd, grad_mask, N, d)

#         t0 = time.time()

#         y_pred = sparse_gp(
#             cov, X[:, :N], X_test,
#             y_train, grad_mask,
#             args.nugget, args.rho,
#             args.k_neighbors,
#             args.type, args.threads
#         )


    for mr2 in hessian_missing_rates:

        grad_mask = generate_grad_mask(d, N, 0.50)
        hess_mask = generate_hessian_mask(6, N, mr2)

        meas = build_measurements(X[:, :N], grad_mask, hess_mask)
        cond = compute_condition_number(cov, meas, args.nugget)

        y_train = build_targets(Y0, Yd, Ydd, grad_mask, hess_mask, N, d)

        y_pred = sparse_gp(
            cov, X[:, :N], X_test,
            y_train, grad_mask, hess_mask,
            args.nugget, args.rho,
            args.k_neighbors,
            args.type, args.threads
        )
        t = time.time() - t0

#         results.append({
#             "N": N,
#             "missing_rate": mr,
#             "mse_sparse": mse(y_test, y_pred),
#             "time_sparse": t,
#             "cond_number": cond
#         })

        results.append({
            "N": N,
            "grad_missing": 0.50,
            "hess_missing": mr2,
            "mse_sparse": mse(y_test, y_pred),
            "time_sparse": t,
            "cond_number": cond
        })

    df = pd.DataFrame(results)
    df.to_csv(f"partial_derivatives_fixed_with_cond_{N}_exp_1.csv", index=False)

    print("Saved: partial_derivatives_fixed_with_cond.csv")

Fixed N = 27
this is P  231
this is meas_train  231
this is P  194
this is meas_train  194
this is P  156
this is meas_train  156
this is P  117
this is meas_train  117
Saved: partial_derivatives_fixed_with_cond.csv


# **Working code for 3D and upto 3rd order-partial observations:**

# **All 0 order available, fixed partial (0.25) 1st order missing, fixed partial (0.25) 2nd order missing, and variable missing 3rd order-with conditioning**

In [1]:
# import numpy as np
# import pandas as pd
# import time
# import argparse

# from Cov import GaussianCovariance_generic
# from Factors import (
#     ExplicitKLFactorization,
#     ImplicitKLFactorization
# )

# from meas import (
#     PointMeasurement,
#     dPointMeasurement, ddPointMeasurement, dddPointMeasurement
# )

# # =========================================================
# # Args
# # =========================================================

# def parse_args():
#     parser = argparse.ArgumentParser()

#     parser.add_argument("--sigma", type=float, default=2.15)
#     parser.add_argument("--h", type=int, default=27)
#     parser.add_argument("--nugget", type=float, default=1e-10)
#     parser.add_argument("--rho", type=float, default=10.0)
#     parser.add_argument("--k_neighbors", type=int, default=1)
#     parser.add_argument("--threads", type=int, default=1)
#     parser.add_argument("--type", type=str, default="points")

#     args, _ = parser.parse_known_args()
#     return args


# # =========================================================
# # Mask
# # =========================================================

# # this mask is for gradient
# def generate_grad_mask(d, N, missing_rate, seed=0):
#     rng = np.random.default_rng(seed)
#     return rng.random((d, N)) > missing_rate


# # this mask is for hessians
# def generate_hessian_mask(n_hess, N, missing_rate, seed=0):
#     rng = np.random.default_rng(seed)
#     return rng.random((n_hess, N)) > missing_rate

# def generate_third_mask(n_comp, N, missing_rate, seed=0):
#     rng = np.random.default_rng(seed)
#     return rng.random((n_comp, N)) > missing_rate

# #remapping
# def remap_idx(t):
#     return tuple(i - 1 for i in t)

# # =========================================================
# # Measurements
# # =========================================================

# def build_measurements(X, grad_mask, hess_mask, third_mask):
#     d, N = X.shape

#     meas_0 = [PointMeasurement(X[:, i]) for i in range(N)]

#     meas_d = [
#         dPointMeasurement(X[:, i], j)
#         for j in range(d)
#         for i in range(N)
#         if grad_mask[j, i]
#     ]
    

#     ind_2=[(0, 0), (0, 1), (1,1), (0,2), (1,2),(2,2)]
#     meas_dd=[]
    
# #     for j in range(len(ind_2)):
# #         for i in range(N):
# #             meas_dd.append(ddPointMeasurement(X[:, i],ind_2[j]))

    
#     for j in range(len(ind_2)):
#         for i in range(N):
#             if hess_mask[j, i]:
#                 meas_dd.append(ddPointMeasurement(X[:, i], ind_2[j]))
                
#     ind_3 = [(1,1,1),(1,1,2),(1,2,2),(2,2,2),(1,1,3),(1,2,3),(2,2,3),(1,3,3),(2,3,3),(3,3,3)]
#     meas_ddd = []
#     for j in range(len(ind_3)):
#         for i in range(N):
#             if third_mask[j, i]:
#                 meas_ddd.append(dddPointMeasurement(X[:, i], ind_3[j]))

#     return meas_0 + meas_d + meas_dd + meas_ddd


# # =========================================================
# # Kernel helpers
# # =========================================================

# def build_symmetric(cov, M):
#     N = len(M)
#     K = np.zeros((N, N))

#     for i in range(N):
#         for j in range(N):
#             K[i, j] = cov(M[i], M[j])

#     return K


# # =========================================================
# # Condition number
# # =========================================================

# def compute_condition_number(cov, measurements, nugget):
#     K = build_symmetric(cov, measurements)
#     K = K + nugget * np.eye(K.shape[0])
#     return np.linalg.cond(K)


# # =========================================================
# # Targets
# # =========================================================

# def build_targets(Y0, Yd, Ydd, Yddd, grad_mask, hess_mask, third_mask, N, d):

# #     y = [Y0]

# #     for j in range(d):
# #         for i in range(N):
# #             if grad_mask[j, i]:
# #                 y.append([Yd[j, i]])

# #     return np.concatenate(y)
#     y = [Y0]
#     for j in range(d):
#         for i in range(N):
#             if grad_mask[j, i]:
#                 y.append([Yd[j, i]])

    
# #     for j in range(Ydd.shape[0]):
# #         for i in range(N):
# #             y.append([Ydd[j, i]])
    
#     for j in range(Ydd.shape[0]):
#         for i in range(N):
#             if hess_mask[j, i]:
#                 y.append([Ydd[j, i]])
                
                
#     for j in range(Yddd.shape[0]):
#         for i in range(N):
#             if third_mask[j, i]:
#                 y.append([Yddd[j, i]])

#     return np.concatenate(y)


# # =========================================================
# # Sparse GP
# # =========================================================

# def sparse_gp(cov, X_train, X_test, y_train,
#               grad_mask, hess_mask, third_mask, nugget, rho, k_neighbors, ordering, threads):

#     d, N = X_train.shape

#     meas_0 = [PointMeasurement(X_train[:, i]) for i in range(N)]

#     meas_d = [
#         dPointMeasurement(X_train[:, i], j)
#         for j in range(d)
#         for i in range(N)
#         if grad_mask[j, i]
#     ]
           
    
#     ind_2=[(0, 0), (0, 1), (1,1), (0,2), (1,2),(2,2)]
#     meas_dd=[]
    
    
# #     for j in range(len(ind_2)):
# #         for i in range(N):
# #             meas_dd.append(ddPointMeasurement(X_train[:, i],ind_2[j]))     
    
#     for j in range(len(ind_2)):
#         for i in range(N):
#             if hess_mask[j, i]:
#                 meas_dd.append(ddPointMeasurement(X_train[:, i], ind_2[j]))
                
                 
#     ind_3 = [(1,1,1),(1,1,2),(1,2,2),(2,2,2),(1,1,3),(1,2,3),(2,2,3),(1,3,3),(2,3,3),(3,3,3)]
#     meas_ddd = []
#     for j in range(len(ind_3)):
#         for i in range(N):
#             if third_mask[j, i]:
#                 meas_ddd.append(dddPointMeasurement(X[:, i], ind_3[j]))
    
    
#     meas = [meas_0, meas_d, meas_dd, meas_ddd]

#     if ordering == "points":
#         implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d(
#             cov, meas, rho, k_neighbors
#         )
#     else:
#         implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(
#             cov, meas, rho, k_neighbors
#         )

#     explicit = ExplicitKLFactorization.Explicit_from_implicit(
#         implicit,
#         nugget=nugget,
#         N_threads=threads
#     )

#     U = explicit.U
#     L = U.transpose().tocsc()
#     P = explicit.P
#     print("this is P ", len(P))

#     rhs = np.zeros_like(y_train)
#     rhs[P] = U @ (L @ y_train[P])

#     meas_train = meas_0 + meas_d + meas_dd + meas_ddd
#     print("this is meas_train ", len(meas_train))
#     reordered = [meas_train[i] for i in P]

#     meas_test = [PointMeasurement(X_test[:, i]) for i in range(X_test.shape[1])]

#     Ktest = np.zeros((len(meas_test), len(reordered)))
#     for i in range(len(meas_test)):
#         for j in range(len(reordered)):
#             Ktest[i, j] = cov(meas_test[i], reordered[j])

#     return Ktest @ rhs[P]


# # =========================================================
# # RMSE
# # =========================================================

# def mse(y_true, y_pred):
#     return np.mean((y_true - y_pred) ** 2)


# # =========================================================
# # MAIN (FIXED N EXPERIMENT + CONDITION NUMBER)
# # =========================================================

# if __name__ == "__main__":

#     args = parse_args()

#     train = pd.read_csv(f"./train_G3_{args.h}.csv")
#     test = pd.read_csv("./test_G3.csv")

#     X = np.vstack([
#         train["X0"].values,
#         train["X1"].values,
#         train["X2"].values
#     ])

#     X_test = np.vstack([
#         test["X0"].values,
#         test["X1"].values,
#         test["X2"].values
#     ])

#     y_test = test["[0]"].values

#     N = args.h
#     d = X.shape[0]

#     Y0 = train["[0]"].values[:N]

#     csv_first_order = ["[1]", "[2]", "[3]"]

#     ind_1 = [remap_idx((i,)) for i in [1,2,3]]
 
    
# #     Yd = np.vstack([
# #         train["[1]"].values[:N],
# #         train["[2]"].values[:N],
# #         train["[3]"].values[:N]
# #     ])
    
#     Yd = np.stack([train[c].values[:N] for c in ind_1])
    
# #     second_order = [
# #         "[1, 1]", "[1, 2]", "[2, 2]", "[1, 3]", "[2, 3]", "[3, 3]"
# #     ]

#     ind_2 = [
#         remap_idx((i, j))
#         for i in range(1, 4)
#         for j in range(i, 4)
#     ]
    
    
#     Ydd = np.vstack([
#         train[c].values[:N]
#         for c in ind_2
#     ])
    
# #     third_order_cols = [
# #         "[1, 1, 1]", "[1, 1, 2]", "[1, 2, 2]", "[2, 2, 2]",
# #         "[1, 1, 3]", "[1, 2, 3]", "[2, 2, 3]",
# #         "[1, 3, 3]", "[2, 3, 3]", "[3, 3, 3]"
# #     ]


#     ind_3 = [
#         remap_idx((i, j, k))
#         for i in range(1, 4)
#         for j in range(i, 4)
#         for k in range(j, 4)
#     ]

#     Yddd = np.vstack([
#         train[c].values[:N]
#         for c in ind_3
#     ])
    
#     cov = GaussianCovariance_generic(args.sigma)

#     results = []

# #     missing_rates = [0.0, 0.25, 0.5, 0.75]

#     third_missing_rates = [0.0, 0.25, 0.5, 0.75]
#     print(f"Fixed N = {N}")

# #     for mr in missing_rates:

# #         grad_mask = generate_grad_mask(d, N, 0.5)

# #         # ---- build measurements for condition number
# #         meas = build_measurements(X[:, :N], grad_mask)
# #         cond = compute_condition_number(cov, meas, args.nugget)

# #         y_train = build_targets(Y0, Yd, Ydd, grad_mask, N, d)

# #         t0 = time.time()

# #         y_pred = sparse_gp(
# #             cov, X[:, :N], X_test,
# #             y_train, grad_mask,
# #             args.nugget, args.rho,
# #             args.k_neighbors,
# #             args.type, args.threads
# #         )

#     for mr3 in third_missing_rates:

#         grad_mask = generate_grad_mask(d, N, 0.75)
#         hess_mask = generate_hessian_mask(6, N, 0.5)
#         third_mask = generate_third_mask(10, N, mr3)

#         meas = build_measurements(X[:, :N], grad_mask, hess_mask, third_mask)
#         cond = compute_condition_number(cov, meas, args.nugget)

#         y_train = build_targets(
#             Y0, Yd, Ydd, Yddd,
#             grad_mask, hess_mask, third_mask,
#             N, d
#         )

#         y_pred = sparse_gp(
#             cov, X[:, :N], X_test,
#             y_train,
#             grad_mask, hess_mask, third_mask,
#             args.nugget, args.rho,
#             args.k_neighbors,
#             args.type, args.threads
#         )
#         t = time.time() - t0

# #         results.append({
# #             "N": N,
# #             "missing_rate": mr,
# #             "mse_sparse": mse(y_test, y_pred),
# #             "time_sparse": t,
# #             "cond_number": cond
# #         })

#         results.append({
#             "N": N,
#             "grad_missing": 0.75,
#             "hess_missing": 0.5,
#             "third_missing": mr3,
#             "mse_sparse": mse(y_test, y_pred),
#             "time_sparse": t,
#             "cond_number": cond
#         })

#     df = pd.DataFrame(results)
#     df.to_csv(f"partial_derivatives_fixed_with_cond_{N}_exp_1.csv", index=False)

#     print("Saved: partial_derivatives_fixed_with_cond.csv")

In [34]:
import numpy as np
import pandas as pd
import time
import argparse

from Cov import GaussianCovariance_generic
from Factors import (
    ExplicitKLFactorization,
    ImplicitKLFactorization
)

from meas import (
    PointMeasurement,
    dPointMeasurement,
    ddPointMeasurement,
    dddPointMeasurement
)

# =========================================================
# Args
# =========================================================

def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--sigma", type=float, default=1.15)
    parser.add_argument("--h", type=int, default=27)
    parser.add_argument("--nugget", type=float, default=1e-6)
    parser.add_argument("--rho", type=float, default=10.0)
    parser.add_argument("--k_neighbors", type=int, default=1)
    parser.add_argument("--threads", type=int, default=1)
    parser.add_argument("--type", type=str, default="points")

    args, _ = parser.parse_known_args()
    return args


# =========================================================
# Masks
# =========================================================

def generate_grad_mask(d, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((d, N)) > missing_rate


def generate_hessian_mask(n_hess, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n_hess, N)) > missing_rate


def generate_third_mask(n_comp, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n_comp, N)) > missing_rate


# =========================================================
# Index remapping (CRITICAL FIX)
# =========================================================

def remap_idx(t):
    return tuple(i - 1 for i in t)


# =========================================================
# Measurements
# =========================================================

def build_measurements(X, grad_mask, hess_mask, third_mask):
    d, N = X.shape

    meas_0 = [PointMeasurement(X[:, i]) for i in range(N)]

    # gradients
    meas_d = [
        dPointMeasurement(X[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]

    # second order (0-based consistent)
    ind_2 = [
        remap_idx((i, j))
        for i in range(1, 4)
        for j in range(i, 4)
    ]

    meas_dd = []
    for j in range(len(ind_2)):
        for i in range(N):
            if hess_mask[j, i]:
                meas_dd.append(ddPointMeasurement(X[:, i], ind_2[j]))

    # third order
    ind_3 = [
        remap_idx((i, j, k))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
    ]

    meas_ddd = []
    for j in range(len(ind_3)):
        for i in range(N):
            if third_mask[j, i]:
                meas_ddd.append(dddPointMeasurement(X[:, i], ind_3[j]))

    return meas_0 + meas_d + meas_dd + meas_ddd


# =========================================================
# Kernel helpers
# =========================================================

def build_symmetric(cov, M):
    N = len(M)
    K = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            K[i, j] = cov(M[i], M[j])

    return K


def compute_condition_number(cov, measurements, nugget):
    K = build_symmetric(cov, measurements)
    K = K + nugget * np.eye(K.shape[0])
    return np.linalg.cond(K)


# =========================================================
# Targets
# =========================================================

def build_targets(Y0, Yd, Ydd, Yddd,
                  grad_mask, hess_mask, third_mask,
                  N, d):

    y = [Y0]

    # first order
    for j in range(d):
        for i in range(N):
            if grad_mask[j, i]:
                y.append([Yd[j, i]])

    # second order
    for j in range(Ydd.shape[0]):
        for i in range(N):
            if hess_mask[j, i]:
                y.append([Ydd[j, i]])

    # third order
    for j in range(Yddd.shape[0]):
        for i in range(N):
            if third_mask[j, i]:
                y.append([Yddd[j, i]])

    return np.concatenate(y)


# =========================================================
# Sparse GP
# =========================================================

def sparse_gp(cov, X_train, X_test, y_train,
              grad_mask, hess_mask, third_mask,
              nugget, rho, k_neighbors, ordering, threads):

    d, N = X_train.shape

    meas_0 = [PointMeasurement(X_train[:, i]) for i in range(N)]

    meas_d = [
        dPointMeasurement(X_train[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]

    ind_2 = [
        remap_idx((i, j))
        for i in range(1, 4)
        for j in range(i, 4)
    ]

    meas_dd = []
    for j in range(len(ind_2)):
        for i in range(N):
            if hess_mask[j, i]:
                meas_dd.append(ddPointMeasurement(X_train[:, i], ind_2[j]))

    ind_3 = [
        remap_idx((i, j, k))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
    ]

    meas_ddd = []
    for j in range(len(ind_3)):
        for i in range(N):
            if third_mask[j, i]:
                meas_ddd.append(dddPointMeasurement(X_train[:, i], ind_3[j]))

    meas = [meas_0, meas_d, meas_dd, meas_ddd]

    if ordering == "points":
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d(
            cov, meas, rho, k_neighbors
        )
    else:
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(
            cov, meas, rho, k_neighbors
        )

    explicit = ExplicitKLFactorization.Explicit_from_implicit(
        implicit,
        nugget=nugget,
        N_threads=threads
    )

    U = explicit.U
    L = U.transpose().tocsc()
    P = explicit.P

    rhs = np.zeros_like(y_train)
    rhs[P] = U @ (L @ y_train[P])

    meas_train = meas_0 + meas_d + meas_dd + meas_ddd
    reordered = [meas_train[i] for i in P]

    meas_test = [PointMeasurement(X_test[:, i]) for i in range(X_test.shape[1])]

    Ktest = np.zeros((len(meas_test), len(reordered)))
    for i in range(len(meas_test)):
        for j in range(len(reordered)):
            Ktest[i, j] = cov(meas_test[i], reordered[j])

    return Ktest @ rhs[P]


# =========================================================
# RMSE
# =========================================================

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    args = parse_args()

    train = pd.read_csv(f"./train_G3_{args.h}.csv")
    test = pd.read_csv("./test_G3.csv")

    X = np.vstack([
        train["X0"].values,
        train["X1"].values,
        train["X2"].values
    ])

    X_test = np.vstack([
        test["X0"].values,
        test["X1"].values,
        test["X2"].values
    ])

    y_test = test["[0]"].values

    N = args.h
    d = X.shape[0]

    Y0 = train["[0]"].values[:N]

    # first order
    Yd = np.vstack([
        train["[1]"].values[:N],
        train["[2]"].values[:N],
        train["[3]"].values[:N]
    ])

    # second order
    second_cols = ["[1, 1]", "[1, 2]", "[2, 2]", "[1, 3]", "[2, 3]", "[3, 3]"]

    Ydd = np.vstack([
        train[c].values[:N] for c in second_cols
    ])

    # third order
    third_cols = [
        "[1, 1, 1]", "[1, 1, 2]", "[1, 2, 2]", "[2, 2, 2]",
        "[1, 1, 3]", "[1, 2, 3]", "[2, 2, 3]",
        "[1, 3, 3]", "[2, 3, 3]", "[3, 3, 3]"
    ]

    Yddd = np.vstack([
        train[c].values[:N] for c in third_cols
    ])

    cov = GaussianCovariance_generic(args.sigma)

    results = []

    third_missing_rates = [0.0, 0.25, 0.5, 0.75]

    print(f"Fixed N = {N}")

    for mr3 in third_missing_rates:

        grad_mask = generate_grad_mask(d, N, 0.25)
        hess_mask = generate_hessian_mask(6, N, 0.25)
        third_mask = generate_third_mask(10, N, mr3)

        meas = build_measurements(X[:, :N], grad_mask, hess_mask, third_mask)
        cond = compute_condition_number(cov, meas, args.nugget)

        y_train = build_targets(
            Y0, Yd, Ydd, Yddd,
            grad_mask, hess_mask, third_mask,
            N, d
        )

        t0 = time.time()

        y_pred = sparse_gp(
            cov, X[:, :N], X_test,
            y_train,
            grad_mask, hess_mask, third_mask,
            args.nugget, args.rho,
            args.k_neighbors,
            args.type, args.threads
        )

        t = time.time() - t0

        results.append({
            "N": N,
            "grad_missing": 0.25,
            "hess_missing": 0.25,
            "third_missing": mr3,
            "mse_sparse": mse(y_test, y_pred),
            "time_sparse": t,
            "cond_number": cond
        })

    df = pd.DataFrame(results)
    df.to_csv(f"partial_derivatives_fixed_with_cond_{N}_exp_1.csv", index=False)

    print("Saved results.")

Fixed N = 27
Saved results.


# **Working code for 3D and upto 4th order-partial observations:**

# **All 0 order available, fixed partial (0.25) 1st order missing, fixed partial (0.25) 2nd order missing, fixed partial (0.25) 3rd order missing and variable 4th ordermissing  order-with conditioning**

In [45]:
import numpy as np
import pandas as pd
import time
import argparse

from Cov import GaussianCovariance_generic
from Factors import (
    ExplicitKLFactorization,
    ImplicitKLFactorization
)

from meas import (
    PointMeasurement,
    dPointMeasurement,
    ddPointMeasurement,
    dddPointMeasurement, ddddPointMeasurement
)

# =========================================================
# Args
# =========================================================

def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--sigma", type=float, default=0.95)
    parser.add_argument("--h", type=int, default=27)
    parser.add_argument("--nugget", type=float, default=1e-5)
    parser.add_argument("--rho", type=float, default=10.0)
    parser.add_argument("--k_neighbors", type=int, default=1)
    parser.add_argument("--threads", type=int, default=1)
    parser.add_argument("--type", type=str, default="points")

    args, _ = parser.parse_known_args()
    return args


# =========================================================
# Masks
# =========================================================

def generate_grad_mask(d, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((d, N)) > missing_rate


def generate_hessian_mask(n_hess, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n_hess, N)) > missing_rate


def generate_third_mask(n_comp, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n_comp, N)) > missing_rate

def generate_fourth_mask(n_comp, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n_comp, N)) > missing_rate


# =========================================================
# Index remapping (CRITICAL FIX)
# =========================================================

def remap_idx(t):
    return tuple(i - 1 for i in t)


# =========================================================
# Measurements
# =========================================================

def build_measurements(X, grad_mask, hess_mask, third_mask, fourth_mask):
    d, N = X.shape

    meas_0 = [PointMeasurement(X[:, i]) for i in range(N)]

    # gradients
    meas_d = [
        dPointMeasurement(X[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]

    # second order (0-based consistent)
    ind_2 = [
        remap_idx((i, j))
        for i in range(1, 4)
        for j in range(i, 4)
    ]

    meas_dd = []
    for j in range(len(ind_2)):
        for i in range(N):
            if hess_mask[j, i]:
                meas_dd.append(ddPointMeasurement(X[:, i], ind_2[j]))

    # third order
    ind_3 = [
        remap_idx((i, j, k))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
    ]

    meas_ddd = []
    for j in range(len(ind_3)):
        for i in range(N):
            if third_mask[j, i]:
                meas_ddd.append(dddPointMeasurement(X[:, i], ind_3[j]))
                
    # foruth order
    ind_4 = [
        remap_idx((i, j, k, l))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
        for l in range(k, 4)
    ]

    meas_dddd = []
    for j in range(len(ind_4)):
        for i in range(N):
            if fourth_mask[j, i]:
                meas_dddd.append(ddddPointMeasurement(X[:, i], ind_4[j]))

    return meas_0 + meas_d + meas_dd + meas_ddd+meas_dddd


# =========================================================
# Kernel helpers
# =========================================================

def build_symmetric(cov, M):
    N = len(M)
    K = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            K[i, j] = cov(M[i], M[j])

    return K


def compute_condition_number(cov, measurements, nugget):
    K = build_symmetric(cov, measurements)
    K = K + nugget * np.eye(K.shape[0])
    return np.linalg.cond(K)


# =========================================================
# Targets
# =========================================================

def build_targets(Y0, Yd, Ydd, Yddd,Ydddd,
                  grad_mask, hess_mask, third_mask, fourth_mask,
                  N, d):

    y = [Y0]

    # first order
    for j in range(d):
        for i in range(N):
            if grad_mask[j, i]:
                y.append([Yd[j, i]])

    # second order
    for j in range(Ydd.shape[0]):
        for i in range(N):
            if hess_mask[j, i]:
                y.append([Ydd[j, i]])

    # third order
    for j in range(Yddd.shape[0]):
        for i in range(N):
            if third_mask[j, i]:
                y.append([Yddd[j, i]])
                
    # fourth order
    for j in range(Ydddd.shape[0]):
        for i in range(N):
            if fourth_mask[j, i]:
                y.append([Ydddd[j, i]])

    return np.concatenate(y)


# =========================================================
# Sparse GP
# =========================================================

def sparse_gp(cov, X_train, X_test, y_train,
              grad_mask, hess_mask, third_mask, fourth_mask,
              nugget, rho, k_neighbors, ordering, threads):

    d, N = X_train.shape

    meas_0 = [PointMeasurement(X_train[:, i]) for i in range(N)]

    meas_d = [
        dPointMeasurement(X_train[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]

    ind_2 = [
        remap_idx((i, j))
        for i in range(1, 4)
        for j in range(i, 4)
    ]

    meas_dd = []
    for j in range(len(ind_2)):
        for i in range(N):
            if hess_mask[j, i]:
                meas_dd.append(ddPointMeasurement(X_train[:, i], ind_2[j]))

    ind_3 = [
        remap_idx((i, j, k))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
    ]

    meas_ddd = []
    for j in range(len(ind_3)):
        for i in range(N):
            if third_mask[j, i]:
                meas_ddd.append(dddPointMeasurement(X_train[:, i], ind_3[j]))
                
                
    
    ind_4 = [
        remap_idx((i, j, k, l))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
        for l in range(k, 4)
    ]

    meas_dddd = []
    for j in range(len(ind_4)):
        for i in range(N):
            if fourth_mask[j, i]:
                meas_dddd.append(ddddPointMeasurement(X_train[:, i], ind_4[j]))

    meas = [meas_0, meas_d, meas_dd, meas_ddd, meas_dddd]

    if ordering == "points":
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d(
            cov, meas, rho, k_neighbors
        )
    else:
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(
            cov, meas, rho, k_neighbors
        )

    explicit = ExplicitKLFactorization.Explicit_from_implicit(
        implicit,
        nugget=nugget,
        N_threads=threads
    )

    U = explicit.U
    L = U.transpose().tocsc()
    P = explicit.P

    rhs = np.zeros_like(y_train)
    rhs[P] = U @ (L @ y_train[P])

    meas_train = meas_0 + meas_d + meas_dd + meas_ddd + meas_dddd
    reordered = [meas_train[i] for i in P]

    meas_test = [PointMeasurement(X_test[:, i]) for i in range(X_test.shape[1])]

    Ktest = np.zeros((len(meas_test), len(reordered)))
    for i in range(len(meas_test)):
        for j in range(len(reordered)):
            Ktest[i, j] = cov(meas_test[i], reordered[j])

    return Ktest @ rhs[P]


# =========================================================
# RMSE
# =========================================================

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    args = parse_args()

    train = pd.read_csv(f"./train_G3_{args.h}.csv")
    test = pd.read_csv("./test_G3.csv")

    X = np.vstack([
        train["X0"].values,
        train["X1"].values,
        train["X2"].values
    ])

    X_test = np.vstack([
        test["X0"].values,
        test["X1"].values,
        test["X2"].values
    ])

    y_test = test["[0]"].values

    N = args.h
    d = X.shape[0]

    Y0 = train["[0]"].values[:N]

    # first order
    Yd = np.vstack([
        train["[1]"].values[:N],
        train["[2]"].values[:N],
        train["[3]"].values[:N]
    ])

    # second order
    second_cols = ["[1, 1]", "[1, 2]", "[2, 2]", "[1, 3]", "[2, 3]", "[3, 3]"]

    Ydd = np.vstack([
        train[c].values[:N] for c in second_cols
    ])

    # third order
    third_cols = [
        "[1, 1, 1]", "[1, 1, 2]", "[1, 2, 2]", "[2, 2, 2]",
        "[1, 1, 3]", "[1, 2, 3]", "[2, 2, 3]",
        "[1, 3, 3]", "[2, 3, 3]", "[3, 3, 3]"
    ]

    Yddd = np.vstack([
        train[c].values[:N] for c in third_cols
    ])
    
    
    
    # fourth order
    fourth_cols = ["[1, 1, 1, 1]","[1, 1, 1, 2]","[1, 1, 2, 2]","[1, 2, 2, 2]","[2, 2, 2, 2]",
                   "[1, 1, 1, 3]","[1, 1, 2, 3]","[1, 2, 2, 3]","[2, 2, 2, 3]","[1, 1, 3, 3]",
                   "[1, 2, 3, 3]","[2, 2, 3, 3]","[1, 3, 3, 3]","[2, 3, 3, 3]","[3, 3, 3, 3]"
    ]

    Ydddd = np.vstack([
        train[c].values[:N] for c in fourth_cols
    ])

    cov = GaussianCovariance_generic(args.sigma)

    results = []

    fourth_missing_rates = [0.0, 0.25, 0.5, 0.75]

    print(f"Fixed N = {N}")

    for mr4 in fourth_missing_rates:

        grad_mask = generate_grad_mask(d, N, 0.5)
        hess_mask = generate_hessian_mask(6, N, 0.5)
        third_mask = generate_third_mask(10, N, 0.5)
        fourth_mask = generate_fourth_mask(15, N, mr4)

        meas = build_measurements(X[:, :N], grad_mask, hess_mask, third_mask, fourth_mask)
        cond = compute_condition_number(cov, meas, args.nugget)

        y_train = build_targets(
            Y0, Yd, Ydd, Yddd, Ydddd,
            grad_mask, hess_mask, third_mask, fourth_mask,
            N, d
        )

        t0 = time.time()

        y_pred = sparse_gp(
            cov, X[:, :N], X_test,
            y_train,
            grad_mask, hess_mask, third_mask, fourth_mask,
            args.nugget, args.rho,
            args.k_neighbors,
            args.type, args.threads
        )

        t = time.time() - t0

        results.append({
            "N": N,
            "grad_missing": 0.5,
            "hess_missing": 0.5,
            "third_missing": 0.5,
            "fourth_missing": mr4,
            "mse_sparse": mse(y_test, y_pred),
            "time_sparse": t,
            "cond_number": cond
        })

    df = pd.DataFrame(results)
    df.to_csv(f"partial_derivatives_fixed_with_cond_{N}_exp_1.csv", index=False)

    print("Saved results.")

Fixed N = 27
Saved results.


# **Working code for 3D and upto 4th order-partial observations:**

# **All 0 order not available (0.25 missing), fixed partial (0.25) 1st order missing, fixed partial (0.25) 2nd order missing, fixed partial (0.25) 3rd order missing and variable 4th order missing  order-with conditioning**

In [47]:
import numpy as np
import pandas as pd
import time
import argparse

from Cov import GaussianCovariance_generic
from Factors import (
    ExplicitKLFactorization,
    ImplicitKLFactorization
)

from meas import (
    PointMeasurement,
    dPointMeasurement,
    ddPointMeasurement,
    dddPointMeasurement, ddddPointMeasurement
)

# =========================================================
# Args
# =========================================================

def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--sigma", type=float, default=0.95)
    parser.add_argument("--h", type=int, default=27)
    parser.add_argument("--nugget", type=float, default=1e-5)
    parser.add_argument("--rho", type=float, default=10.0)
    parser.add_argument("--k_neighbors", type=int, default=1)
    parser.add_argument("--threads", type=int, default=1)
    parser.add_argument("--type", type=str, default="points")

    args, _ = parser.parse_known_args()
    return args


# =========================================================
# Masks
# =========================================================
def generate_point_mask(N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random(N) > missing_rate


def generate_grad_mask(d, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((d, N)) > missing_rate


def generate_hessian_mask(n_hess, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n_hess, N)) > missing_rate


def generate_third_mask(n_comp, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n_comp, N)) > missing_rate

def generate_fourth_mask(n_comp, N, missing_rate, seed=0):
    rng = np.random.default_rng(seed)
    return rng.random((n_comp, N)) > missing_rate


# =========================================================
# Index remapping (CRITICAL FIX)
# =========================================================

def remap_idx(t):
    return tuple(i - 1 for i in t)


# =========================================================
# Measurements
# =========================================================

def build_measurements(X, point_mask, grad_mask, hess_mask, third_mask, fourth_mask):
    d, N = X.shape

    meas_0 = [PointMeasurement(X[:, i]) for i in range(N) if point_mask[i]]

    # gradients
    meas_d = [
        dPointMeasurement(X[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]

    # second order (0-based consistent)
    ind_2 = [
        remap_idx((i, j))
        for i in range(1, 4)
        for j in range(i, 4)
    ]

    meas_dd = []
    for j in range(len(ind_2)):
        for i in range(N):
            if hess_mask[j, i]:
                meas_dd.append(ddPointMeasurement(X[:, i], ind_2[j]))

    # third order
    ind_3 = [
        remap_idx((i, j, k))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
    ]

    meas_ddd = []
    for j in range(len(ind_3)):
        for i in range(N):
            if third_mask[j, i]:
                meas_ddd.append(dddPointMeasurement(X[:, i], ind_3[j]))
                
    # foruth order
    ind_4 = [
        remap_idx((i, j, k, l))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
        for l in range(k, 4)
    ]

    meas_dddd = []
    for j in range(len(ind_4)):
        for i in range(N):
            if fourth_mask[j, i]:
                meas_dddd.append(ddddPointMeasurement(X[:, i], ind_4[j]))

    return meas_0 + meas_d + meas_dd + meas_ddd+meas_dddd


# =========================================================
# Kernel helpers
# =========================================================

def build_symmetric(cov, M):
    N = len(M)
    K = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            K[i, j] = cov(M[i], M[j])

    return K


def compute_condition_number(cov, measurements, nugget):
    K = build_symmetric(cov, measurements)
    K = K + nugget * np.eye(K.shape[0])
    return np.linalg.cond(K)


# =========================================================
# Targets
# =========================================================

def build_targets(Y0, Yd, Ydd, Yddd,Ydddd, point_mask,
                  grad_mask, hess_mask, third_mask, fourth_mask,
                  N, d):
    
    y = []

    for i in range(N):
        if point_mask[i]:
            y.append([Y0[i]])

    # first order
    for j in range(d):
        for i in range(N):
            if grad_mask[j, i]:
                y.append([Yd[j, i]])

    # second order
    for j in range(Ydd.shape[0]):
        for i in range(N):
            if hess_mask[j, i]:
                y.append([Ydd[j, i]])

    # third order
    for j in range(Yddd.shape[0]):
        for i in range(N):
            if third_mask[j, i]:
                y.append([Yddd[j, i]])
                
    # fourth order
    for j in range(Ydddd.shape[0]):
        for i in range(N):
            if fourth_mask[j, i]:
                y.append([Ydddd[j, i]])

    return np.concatenate(y)


# =========================================================
# Sparse GP
# =========================================================

def sparse_gp(cov, X_train, X_test, y_train, point_mask,
              grad_mask, hess_mask, third_mask, fourth_mask,
              nugget, rho, k_neighbors, ordering, threads):

    d, N = X_train.shape

    meas_0 = [PointMeasurement(X_train[:, i]) for i in range(N) if point_mask[i]]

    meas_d = [
        dPointMeasurement(X_train[:, i], j)
        for j in range(d)
        for i in range(N)
        if grad_mask[j, i]
    ]

    ind_2 = [
        remap_idx((i, j))
        for i in range(1, 4)
        for j in range(i, 4)
    ]

    meas_dd = []
    for j in range(len(ind_2)):
        for i in range(N):
            if hess_mask[j, i]:
                meas_dd.append(ddPointMeasurement(X_train[:, i], ind_2[j]))

    ind_3 = [
        remap_idx((i, j, k))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
    ]

    meas_ddd = []
    for j in range(len(ind_3)):
        for i in range(N):
            if third_mask[j, i]:
                meas_ddd.append(dddPointMeasurement(X_train[:, i], ind_3[j]))
                
                
    
    ind_4 = [
        remap_idx((i, j, k, l))
        for i in range(1, 4)
        for j in range(i, 4)
        for k in range(j, 4)
        for l in range(k, 4)
    ]

    meas_dddd = []
    for j in range(len(ind_4)):
        for i in range(N):
            if fourth_mask[j, i]:
                meas_dddd.append(ddddPointMeasurement(X_train[:, i], ind_4[j]))

    meas = [meas_0, meas_d, meas_dd, meas_ddd, meas_dddd]

    if ordering == "points":
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d(
            cov, meas, rho, k_neighbors
        )
    else:
        implicit, _, _ = ImplicitKLFactorization.implicit_kl_factorization_for_d_V2(
            cov, meas, rho, k_neighbors
        )

    explicit = ExplicitKLFactorization.Explicit_from_implicit(
        implicit,
        nugget=nugget,
        N_threads=threads
    )

    U = explicit.U
    L = U.transpose().tocsc()
    P = explicit.P

    rhs = np.zeros_like(y_train)
    rhs[P] = U @ (L @ y_train[P])

    meas_train = meas_0 + meas_d + meas_dd + meas_ddd + meas_dddd
    reordered = [meas_train[i] for i in P]

    meas_test = [PointMeasurement(X_test[:, i]) for i in range(X_test.shape[1])]

    Ktest = np.zeros((len(meas_test), len(reordered)))
    for i in range(len(meas_test)):
        for j in range(len(reordered)):
            Ktest[i, j] = cov(meas_test[i], reordered[j])

    return Ktest @ rhs[P]


# =========================================================
# RMSE
# =========================================================

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    args = parse_args()

    train = pd.read_csv(f"./train_G3_{args.h}.csv")
    test = pd.read_csv("./test_G3.csv")

    X = np.vstack([
        train["X0"].values,
        train["X1"].values,
        train["X2"].values
    ])

    X_test = np.vstack([
        test["X0"].values,
        test["X1"].values,
        test["X2"].values
    ])

    y_test = test["[0]"].values

    N = args.h
    d = X.shape[0]

    Y0 = train["[0]"].values[:N]

    # first order
    Yd = np.vstack([
        train["[1]"].values[:N],
        train["[2]"].values[:N],
        train["[3]"].values[:N]
    ])

    # second order
    second_cols = ["[1, 1]", "[1, 2]", "[2, 2]", "[1, 3]", "[2, 3]", "[3, 3]"]

    Ydd = np.vstack([
        train[c].values[:N] for c in second_cols
    ])

    # third order
    third_cols = [
        "[1, 1, 1]", "[1, 1, 2]", "[1, 2, 2]", "[2, 2, 2]",
        "[1, 1, 3]", "[1, 2, 3]", "[2, 2, 3]",
        "[1, 3, 3]", "[2, 3, 3]", "[3, 3, 3]"
    ]

    Yddd = np.vstack([
        train[c].values[:N] for c in third_cols
    ])
    
    
    
    # fourth order
    fourth_cols = ["[1, 1, 1, 1]","[1, 1, 1, 2]","[1, 1, 2, 2]","[1, 2, 2, 2]","[2, 2, 2, 2]",
                   "[1, 1, 1, 3]","[1, 1, 2, 3]","[1, 2, 2, 3]","[2, 2, 2, 3]","[1, 1, 3, 3]",
                   "[1, 2, 3, 3]","[2, 2, 3, 3]","[1, 3, 3, 3]","[2, 3, 3, 3]","[3, 3, 3, 3]"
    ]

    Ydddd = np.vstack([
        train[c].values[:N] for c in fourth_cols
    ])

    cov = GaussianCovariance_generic(args.sigma)

    results = []

    fourth_missing_rates = [0.0, 0.25, 0.5, 0.75]

    print(f"Fixed N = {N}")

    for mr4 in fourth_missing_rates:
        
        point_mask = generate_point_mask(N, 0.25)
        grad_mask = generate_grad_mask(d, N, 0.5)
        hess_mask = generate_hessian_mask(6, N, 0.5)
        third_mask = generate_third_mask(10, N, 0.5)
        fourth_mask = generate_fourth_mask(15, N, mr4)

        meas = build_measurements(X[:, :N], point_mask, grad_mask, hess_mask, third_mask, fourth_mask)
        cond = compute_condition_number(cov, meas, args.nugget)

        y_train = build_targets(
            Y0, Yd, Ydd, Yddd, Ydddd, point_mask,
            grad_mask, hess_mask, third_mask, fourth_mask,
            N, d
        )

        t0 = time.time()

        y_pred = sparse_gp(
            cov, X[:, :N], X_test,
            y_train, point_mask,
            grad_mask, hess_mask, third_mask, fourth_mask,
            args.nugget, args.rho,
            args.k_neighbors,
            args.type, args.threads
        )

        t = time.time() - t0

        results.append({
            "N": N,
            "point_missing": 0.25,
            "grad_missing": 0.5,
            "hess_missing": 0.5,
            "third_missing": 0.5,
            "fourth_missing": mr4,
            "mse_sparse": mse(y_test, y_pred),
            "time_sparse": t,
            "cond_number": cond
        })

    df = pd.DataFrame(results)
    df.to_csv(f"partial_derivatives_fixed_with_cond_{N}_exp_1.csv", index=False)

    print("Saved results.")

Fixed N = 27
Saved results.
